<a href="https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w05_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/omsawarkar/omsawarkar/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use logistic regression to produce probabilities for a ranked review queue. It is a simple, interpretable starting method for a binary observed-outcome proxy.

My diagnostic target is whether an item’s CTR during 21–31 March 2026 falls below its position-and-volume peer-median CTR established using 1–20 March. The benchmark is frozen before examining later outcomes. This measures later below-benchmark CTR, not human-confirmed review value or the benefit of a content update.

The model will use the five early-window features defined in Week 3: search impressions, search clicks, search CTR, website sessions, and engagement rate. Missing-value handling, transformations, and scaling will be fitted using training items only.

I will compare the model’s probabilities with the unchanged Week 4 peer-gap scoring formula on the same eligible test items. The primary comparison metric will be Precision@10, with the test-set proxy base rate reported alongside it. Average precision will provide an additional ranking summary.

This is a new diagnostic evaluation of the Week 4 baseline. It does not replace or establish the original human-review success measure. A model win is not assumed; weak or negative results will be reported.

In [7]:
%pip -q install huggingface_hub duckdb scikit-learn

import pandas as pd
import numpy as np
import duckdb

from google.colab import userdata
from huggingface_hub import hf_hub_download

# Read the existing token privately.
token = userdata.get("HF_TOKEN").strip()

march_path = hf_hub_download(
    repo_id="FlyRank/internship-warehouse",
    filename=(
        "fact_content_daily_performance/"
        "month=2026-03/data_0.parquet"
    ),
    repo_type="dataset",
    token=token
)

con = duckdb.connect()
con.read_parquet(
    march_path,
    hive_partitioning=True
).create_view("march_raw")

print("March dataset ready.")
display(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM march_raw
""").df())


March dataset ready.


,total_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [8]:
# Reconstruct the Week 4 early-window table.
early = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS observed_days,
        SUM(gsc_impressions) AS impressions,
        SUM(gsc_clicks) AS clicks,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS ctr,
        SUM(ga4_sessions) AS sessions,
        SUM(ga4_engaged_sessions) * 1.0
            / NULLIF(SUM(ga4_sessions), 0) AS engagement_rate,

        SUM(
            CASE WHEN gsc_avg_position > 0
                 THEN gsc_avg_position * gsc_impressions END
        ) / NULLIF(
            SUM(
                CASE WHEN gsc_avg_position > 0
                     THEN gsc_impressions END
            ), 0
        ) AS avg_position

    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-01'
                          AND DATE '2026-03-20'
      AND gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
    HAVING COUNT(*) >= 10
       AND SUM(gsc_impressions) > 0
""").df()

early["position_bucket"] = pd.cut(
    early["avg_position"],
    bins=[0, 3, 10, 20, float("inf")],
    labels=["1–3", ">3–10", ">10–20", ">20"]
)

early["volume_bucket"] = pd.cut(
    early["impressions"],
    bins=[99, 499, 1999, float("inf")],
    labels=["100–499", "500–1,999", "2,000+"]
)

peer_stats = (
    early.loc[
        (early["impressions"] >= 100)
        & early["position_bucket"].notna()
    ]
    .groupby(
        ["position_bucket", "volume_bucket"],
        observed=True
    )
    .agg(
        peer_count=("ctr", "size"),
        peer_median_ctr=("ctr", "median")
    )
    .reset_index()
)

early = early.merge(
    peer_stats,
    on=["position_bucket", "volume_bucket"],
    how="left",
    validate="many_to_one"
)

eligible = (
    (early["impressions"] >= 100)
    & early["position_bucket"].notna()
    & (early["peer_count"] >= 20)
    & early["ctr"].notna()
    & early["peer_median_ctr"].notna()
)

early["baseline_score"] = np.where(
    eligible,
    early["impressions"]
    * (early["peer_median_ctr"] - early["ctr"]).clip(lower=0),
    0.0
)

early["baseline_eligible"] = eligible

assert not early.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()

print("Early-window items:", len(early))
print("Positive baseline scores:", (early["baseline_score"] > 0).sum())
print("Items meeting baseline scoring requirements:", eligible.sum())

Early-window items: 6607
Positive baseline scores: 3280
Items meeting baseline scoring requirements: 6565


In [9]:
# Later observations are used ONLY to construct the outcome.
late = con.sql("""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(*) AS outcome_days,
        SUM(gsc_impressions) AS late_impressions,
        SUM(gsc_clicks) * 1.0
            / NULLIF(SUM(gsc_impressions), 0) AS late_ctr
    FROM march_raw
    WHERE report_date BETWEEN DATE '2026-03-21'
                          AND DATE '2026-03-31'
      AND gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

experiment = early.loc[
    early["baseline_eligible"]
].merge(
    late,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
    validate="one_to_one"
)

# Require enough observed outcome data.
experiment = experiment.loc[
    (experiment["outcome_days"] >= 5)
    & (experiment["late_impressions"] >= 100)
    & experiment["late_ctr"].notna()
].copy()

experiment["later_below_peer"] = (
    experiment["late_ctr"] < experiment["peer_median_ctr"]
).astype(int)

# Stable ordering makes later tie-breaking reproducible.
experiment = experiment.sort_values(
    ["client_hash_id", "content_hash_id"]
).reset_index(drop=True)

feature_columns = [
    "impressions",
    "clicks",
    "ctr",
    "sessions",
    "engagement_rate"
]

assert not experiment.duplicated(
    ["client_hash_id", "content_hash_id"]
).any()

print("Items with usable later outcomes:", len(experiment))
print("Clients:", experiment["client_hash_id"].nunique())
print("Proxy positive rate:", round(
    experiment["later_below_peer"].mean(), 4
))

display(
    experiment["later_below_peer"]
    .value_counts()
    .sort_index()
    .rename_axis("later_below_peer")
    .to_frame("item_count")
)

print("Missing model inputs:")
display(
    experiment[feature_columns]
    .isna().sum().to_frame("missing_count")
)

Items with usable later outcomes: 6531
Clients: 21
Proxy positive rate: 0.6567


,item_count
later_below_peer,
0,2242
1,4289


Missing model inputs:


,missing_count
impressions,0
clicks,0
ctr,0
sessions,0
engagement_rate,0


## 2. Split design

Features and baseline scores use only 1–20 March 2026. Outcome labels use 21–31 March. Items require at least 5 GSC-available outcome days and 100 outcome impressions. This outcome-coverage filter defines the evaluable population; it would not be knowable when initially issuing recommendations.

The resulting comparison dataset has 6,531 items across 21 clients, with 4,289 positive and 2,242 negative proxy labels.

I will use one client-grouped split with test_size=0.25 and random_state=42. No client’s items will appear in both training and test sets. This fraction applies to client groups, so the test fraction of items may differ substantially from 25%.

Preprocessing and logistic regression will be fitted on training items only. The five input columns are explicitly selected; IDs, peer benchmarks, baseline scores, and later measurements are excluded from model inputs.

The unchanged Week 4 peer benchmark uses the full early-window reference population, including early observations from held-out clients. This is a known-reference-population experiment, not fully inductive validation on entirely unseen clients. No later test outcomes are used to calculate the benchmark or fit the model.

The baseline and model will be evaluated on exactly the same test items. This single March split is exploratory; June remains untouched, and I will not change the split or settings to improve the reported result.

In [10]:
from sklearn.model_selection import GroupShuffleSplit

X = experiment[feature_columns].copy()
y = experiment["later_below_peer"].astype(int)
groups = experiment["client_hash_id"]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42
)

train_idx, test_idx = next(splitter.split(X, y, groups))

train_clients = set(groups.iloc[train_idx])
test_clients = set(groups.iloc[test_idx])

assert train_clients.isdisjoint(test_clients)
assert y.iloc[train_idx].nunique() == 2
assert y.iloc[test_idx].nunique() == 2

print("Training items:", len(train_idx))
print("Test items:", len(test_idx))
print("Training clients:", len(train_clients))
print("Test clients:", len(test_clients))
print("Training positive rate:", round(y.iloc[train_idx].mean(), 4))
print("Test positive rate:", round(y.iloc[test_idx].mean(), 4))
print("Client overlap:", len(train_clients & test_clients))


Training items: 6285
Test items: 246
Training clients: 15
Test clients: 6
Training positive rate: 0.6554
Test positive rate: 0.6911
Client overlap: 0


## 3. Train + compare vs my baseline

Both methods were evaluated on the same 246 test items from 6 held-out clients. The test proxy-positive rate was 0.6911.

The Week 4 baseline and logistic regression both achieved Precision@10 of 1.00. At Precision@20, the baseline achieved 1.00 and logistic regression achieved 0.95. Logistic regression had higher average precision (0.8648 versus 0.8399) and ROC-AUC (0.7711 versus 0.7321).

The model therefore did not improve the primary metric. Its better overall ranking summaries do not justify replacing the baseline for the top-10 review task on this evidence. The test set is small, and no statistical significance or broader generalization is claimed.

These metrics evaluate later below-benchmark CTR. They do not validate human review value or the benefit of changing content.

In [11]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import FunctionTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
import sklearn

# All five inputs are non-negative.
assert (X.dropna() >= 0).all().all()

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("log_transform", FunctionTransformer(np.log1p)),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=2000,
        random_state=42
    ))
])

# Fit preprocessing and model on training items only.
model.fit(X.iloc[train_idx], y.iloc[train_idx])

model_scores = model.predict_proba(
    X.iloc[test_idx]
)[:, 1]

baseline_scores = experiment.iloc[test_idx][
    "baseline_score"
].to_numpy()

y_test = y.iloc[test_idx].to_numpy()
test_base_rate = y_test.mean()

def precision_at_k(labels, scores, k):
    k = min(k, len(labels))
    # Stable ties follow the pre-established ID ordering.
    order = np.argsort(-np.asarray(scores), kind="stable")
    return float(np.asarray(labels)[order[:k]].mean())

rows = []

for name, scores in [
    ("Week 4 peer-gap baseline", baseline_scores),
    ("Logistic regression", model_scores)
]:
    rows.append({
        "method": name,
        "test_items": len(y_test),
        "proxy_base_rate": test_base_rate,
        "precision_at_10": precision_at_k(y_test, scores, 10),
        "precision_at_20": precision_at_k(y_test, scores, 20),
        "average_precision": average_precision_score(y_test, scores),
        "roc_auc": roc_auc_score(y_test, scores)
    })

comparison = pd.DataFrame(rows)

print("Same test items and labels for both methods.")
print("scikit-learn version:", sklearn.__version__)
display(comparison.round(4))

print("Logistic-regression solver iterations:")
print(model.named_steps["classifier"].n_iter_)


Same test items and labels for both methods.
scikit-learn version: 1.6.1


,method,test_items,proxy_base_rate,precision_at_10,precision_at_20,average_precision,roc_auc
0,Week 4 peer-gap baseline,246,0.6911,1.0,1.00,0.8399,0.7321
1,Logistic regression,246,0.6911,1.0,0.95,0.8648,0.7711


Logistic-regression solver iterations:
[9]


## 4. Errors and interpretation

### What the model uses

After log transformation and standardization, the largest absolute coefficients are clicks (-0.9392), impressions (+0.5299), and sessions (-0.4118). Holding the other transformed inputs fixed, more clicks and sessions are associated with lower predicted probability of later below-benchmark CTR, while more impressions are associated with higher probability.

These directions are plausible measures of traffic and click performance, but they are not causal explanations. Clicks, impressions, and CTR are mathematically related, so correlated inputs complicate interpretation of individual coefficients.

### Three wrong cases

At the fixed 0.5 threshold, the model makes 68 errors among 246 test items. Its three most confident errors are false negatives:

- Case 8: Predicted probability 0.04440, but the observed label is 1. Early CTR was 0.02625, while late CTR was 0.00403, below its peer benchmark of 0.00693.
- Case 28: Predicted probability 0.10399, but the observed label is 1. Early CTR was 0.03343, while late CTR was 0.00464, below its peer benchmark of 0.00612.
- Case 25: Predicted probability 0.13605, but the observed label is 1. Early CTR was 0.03201, while late CTR was 0.00701, below its peer benchmark of 0.00998.

The CTR values above are fractions. All three cases had relatively strong early CTR followed by substantially lower late CTR. The five early aggregates did not capture those later changes. The available data does not establish why they happened.

### Where errors concentrate

Error rates are 41.18% for positions 1–3 (51 items), 22.29% for >3–10 (157 items), 14.81% for >10–20 (27 items), and 72.73% for >20 (11 items). The last group is particularly small, so its rate is unstable.

These classification errors supplement the ranking evaluation; the 0.5 threshold is not how the top-10 list is selected.

### Decision and limitations

I will retain the Week 4 baseline for now. Logistic regression ties its Precision@10, performs worse at Precision@20, and improves only the broader ranking summaries in this experiment.

The evaluation uses a single month, one

In [12]:
# Coefficients describe associations after log transformation and scaling.
coefficients = pd.DataFrame({
    "feature": feature_columns,
    "coefficient": model.named_steps["classifier"].coef_[0]
})
coefficients["absolute_coefficient"] = (
    coefficients["coefficient"].abs()
)
coefficients = coefficients.sort_values(
    "absolute_coefficient", ascending=False
)

print("Model coefficients — not causal effects:")
display(coefficients.round(4))

# Inspect errors using a fixed 0.5 threshold.
test_review = experiment.iloc[test_idx].copy().reset_index(drop=True)
test_review["model_probability"] = model_scores
test_review["predicted_label"] = (model_scores >= 0.5).astype(int)
test_review["wrong"] = (
    test_review["predicted_label"]
    != test_review["later_below_peer"]
)
test_review["error_magnitude"] = abs(
    test_review["model_probability"]
    - test_review["later_below_peer"]
)

print("Errors at threshold 0.5:", test_review["wrong"].sum())

# Anonymous case numbers refer to this fixed test-table order.
test_review["case_number"] = np.arange(1, len(test_review) + 1)

wrong_cases = (
    test_review.loc[test_review["wrong"]]
    .sort_values("error_magnitude", ascending=False, kind="stable")
    .head(3)
)

print("Three most confident wrong cases:")
display(wrong_cases[[
    "case_number",
    "impressions",
    "clicks",
    "ctr",
    "sessions",
    "engagement_rate",
    "peer_median_ctr",
    "late_ctr",
    "later_below_peer",
    "model_probability",
    "predicted_label"
]].round(5))

print("Errors by early position bucket:")
display(
    test_review.groupby("position_bucket", observed=True)
    .agg(
        test_items=("wrong", "size"),
        error_rate=("wrong", "mean"),
        proxy_positive_rate=("later_below_peer", "mean")
    ).round(4)
)


Model coefficients — not causal effects:


,feature,coefficient,absolute_coefficient
1,clicks,-0.9392,0.9392
0,impressions,0.5299,0.5299
3,sessions,-0.4118,0.4118
2,ctr,-0.2769,0.2769
4,engagement_rate,-0.0247,0.0247


Errors at threshold 0.5: 68
Three most confident wrong cases:


,case_number,impressions,clicks,ctr,sessions,engagement_rate,peer_median_ctr,late_ctr,later_below_peer,model_probability,predicted_label
7,8,15466.0,406.0,0.02625,335.0,0.00000,0.00693,0.00403,1,0.04440,0
27,28,2094.0,70.0,0.03343,73.0,0.02740,0.00612,0.00464,1,0.10399,0
24,25,1437.0,46.0,0.03201,61.0,0.03279,0.00996,0.00701,1,0.13605,0


Errors by early position bucket:


,test_items,error_rate,proxy_positive_rate
position_bucket,,,
1–3,51,0.4118,0.6667
>3–10,157,0.2229,0.7134
>10–20,27,0.1481,0.7778
>20,11,0.7273,0.2727


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.